# tuned - Kaggle smoke run
Prereqs: phone-verified account, Accelerator = **GPU T4 x2** (never P100), Internet **On**,
`HF_TOKEN` added under Add-ons -> Secrets. Set `MODE` below, then
Run All (SAVETEST interactively first; SMOKE via *Save & Run All* in the background).

Optional but recommended: run `notebooks/stage_model.ipynb` once (CPU, zero GPU quota)
and attach its output as an Input here (**+ Add Input -> Your Work**) - the model then
loads from local disk instead of downloading ~7 GiB from HF every session.

Optional: a `WANDB_API_KEY` secret (Add-ons -> Secrets, attached) turns on live
Weights & Biases metrics for the training cell - the one phase batch mode keeps
blind. No secret = no W&B, identical to the old behavior.

## After a re-import
Importing this file creates a **fresh kernel id**, and Kaggle silently drops every
per-notebook attachment with it (hit twice). Before the first run, re-attach ALL of:
1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons -> Secrets: `HF_TOKEN` (required) and `WANDB_API_KEY` (optional) - adding
   a secret is not enough, its **attach checkbox** must be ticked for THIS notebook.
3. **+ Add Input -> Your Work** -> the `stage_model` output dataset (else the run
   falls back to the ~7 GiB hub download - the v6-v9 stall class).

The preflight cell below verifies all of this in seconds and stops before any GPU
quota is spent; `preflight ok` means the attachments survived.

In [ ]:
MODE = "SMOKE"     # PROBE (2-step seq-ceiling check, no Hub) | SAVETEST (4-step save/push gate) | SMOKE (60 steps) | RESUME | MAIN / MAIN_RESUME (production: train.main - needs the derived max_steps committed and data/law_v1.jsonl built)
PROBE_SEQ = 12288   # PROBE only: the seq being requalified (also sizes the probe dataset)

# One lane, no switches: Qwen3-8B under 2x T4 data-parallel via torchrun at
# seq 8192. All four gates green 2026-08-08 (PROBE 12.80/13.00 GiB, SAVETEST,
# SMOKE 60/60 at 74.7 s/step with peaks 12.98/13.18 GiB, RESUME).
# Temporary experiment arms: point CONFIG at an experiment yaml with
# MODE="SMOKE"; each arm pushes to its own ckpt repo and gets its own W&B
# run-name suffix from the config filename. NEVER --resume across configs.
# Adapter-scale arms (rsLoRA 5.66x/11.3x, plain alpha-64) all ran and were
# rejected 2026-08-10/11: loss identical to baseline at every scale, only
# gradient heat changes - see the lora comment in the config.
CONFIG = "configs/law_v1_8b_ddp.yaml"

import os, subprocess

# Every rank must see BOTH GPUs (rank N places itself on cuda:N); a leaked
# single-GPU mask kills rank 1 with "invalid device ordinal" mid-load
# (2026-08-06 crash - sft.py fails fast on it now).
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# 32 chunks, not 16: the transient scales with seq, and ~0.7 GiB at 8192/16 becomes ~1.05 GiB at 12288.
os.environ["UNSLOTH_CE_LOSS_N_CHUNKS"] = "32"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # fragmentation headroom, ~1.4 GiB from the cap
os.environ["HF_HOME"] = "/tmp/hf_cache"          # scratch, NOT the 20GB persisted /kaggle/working
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"    # must be PRESENT and "0": unsloth_zoo force-enables
# hf_transfer when the var is absent; that fast path has no retry/resume AND bypasses xet.
# Do NOT set UNSLOTH_STABLE_DOWNLOADS / HF_HUB_DISABLE_XET: v1-v5 downloaded fine on the
# default xet path; the stalls started exactly when v6 disabled xet, which forces the
# legacy bridge for a xet-backed repo - the very path that hangs from Kaggle.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1" # \r progress spam never renders in batch logs

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpus)
assert gpus.count("T4") == 2, "Expected 2x T4 - Settings -> Accelerator -> 'GPU T4 x2'"
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
# Re-import preflight - importing the notebook file creates a FRESH kernel,
# and Kaggle silently drops attached Secrets and Input mounts with it (hit
# twice, 2026-08-07). Everything here fails in seconds, BEFORE the
# clone/install/download phases spend minutes discovering it the hard way.
import socket
from pathlib import Path

try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError as exc:
    raise SystemExit("no network - Settings -> Internet -> On") from exc

_inp = Path("/kaggle/input")
_missing = []
# same depth-bounded hunt as the token cell below - rglob would crawl every
# attached dataset over network FS
if not any(True for pat in ("token.txt", "*/token.txt", "*/*/token.txt") for _ in _inp.glob(pat)):
    try:
        from kaggle_secrets import UserSecretsClient

        UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        _missing.append("HF_TOKEN - Add-ons -> Secrets: add it AND tick its attach checkbox")
try:
    from kaggle_secrets import UserSecretsClient

    UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    print("note: WANDB_API_KEY not attached - W&B live metrics stay off (optional)")
if not any(_inp.glob("*/qwen3-8b-staged/REVISION.txt")):
    print("WARNING: staged snapshot not attached (+ Add Input -> Your Work -> the")
    print("         stage_model output) - the run will fall back to the ~7 GiB hub")
    print("         download, the exact stall class that killed sessions v6-v9.")
if _missing:
    raise SystemExit("re-attach before running: " + "; ".join(_missing))
print("preflight ok")

In [ ]:
%cd /tmp
!rm -rf /tmp/tuned
!git clone --depth 1 https://github.com/Anant-T/Tuned /tmp/tuned
%cd /tmp/tuned

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
r = subprocess.run(["uv", "pip", "install", "--system", "-e", ".[dev,train]"])
assert r.returncode == 0, "dependency install failed - do not continue on a broken env"

In [ ]:
import os
from itertools import islice
from pathlib import Path

# Depth-bounded search, NOT rglob: Kaggle nests mounts one or two levels deep
# (/kaggle/input/<slug>/[datasets/...]/token.txt - the 3fb2a2d lesson), but
# rglob walks EVERY attached dataset. Attaching a multi-GB dataset (e.g. a
# staged model snapshot) would turn the token hunt into a minutes-long
# network-FS crawl. The diagnostic below streams lazily for the same reason.
inp = Path("/kaggle/input")
tok = None
hits = [*inp.glob("token.txt"), *inp.glob("*/token.txt"), *inp.glob("*/*/token.txt")]
if hits:
    tok = hits[0].read_text().strip()
else:
    sample = [str(p) for p in islice(inp.rglob("*"), 20)] if inp.exists() else "no /kaggle/input"
    print(f"token.txt not found at depth <=2 under /kaggle/input; tree sample: {sample}")
    try:
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise SystemExit(
            "No HF token available: dataset 'tuned-token' not mounted AND "
            f"UI secret unavailable ({type(exc).__name__})."
        ) from exc
os.environ["HF_TOKEN"] = tok
print("HF token loaded (not printed).")

# Optional W&B (Add-ons -> Secrets -> WANDB_API_KEY, attached to this
# notebook): sft.py flips report_to to "wandb" when the key is present, giving
# live loss/grad_norm during the training cell - the one phase Kaggle batch
# keeps blind (output flushes only when a cell COMPLETES). The env vars reach
# the torchrun children via the supervisor's child_env. Missing/unattached
# secret = logging off, exactly the pre-W&B behavior - never a failure.
# The run name keys off CONFIG too: experiment configs reuse MODE="SMOKE",
# and without a suffix every arm would land in W&B as the same
# "8b-ddp-smoke". The suffix is whatever the config filename adds to the
# production stem ("" for the production yaml) - e.g. law_v1_8b_ddp_alpha64
# -> 8b-ddp-smoke-alpha64. This cell runs BEFORE the re-home cell rewrites
# CONFIG to the session-local law_v1_run.yaml copy - keep it that way, or
# every experiment would be named "-run".
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ.setdefault("WANDB_PROJECT", "tuned")
    _variant = CONFIG.rsplit("/", 1)[-1].removesuffix(".yaml").removeprefix("law_v1_8b_ddp").strip("_").replace("_", "-")
    os.environ.setdefault("WANDB_NAME", f"8b-ddp-{MODE.lower()}" + (f"-{_variant}" if _variant else ""))
    print(f"W&B key loaded - live metrics on (run name {os.environ['WANDB_NAME']}).")
except Exception as exc:
    print(f"no WANDB_API_KEY secret ({type(exc).__name__}) - W&B logging off.")

In [ ]:
from importlib.metadata import version

for pkg in ("torch", "transformers", "trl", "unsloth", "bitsandbytes", "peft", "hf_transfer"):
    try:
        print(f"{pkg}=={version(pkg)}")
    except Exception:
        print(f"{pkg}: NOT INSTALLED")

import subprocess

assert subprocess.run(["python", "-m", "pytest", "tests/", "-q"]).returncode == 0, "tests failed - fix before burning GPU quota"

In [ ]:
# Re-home the checkpoint repo to THIS account's HF namespace: the committed
# configs pin the maintainer's, which another account's token cannot push to
# (the 403 create_repo lesson). Session-local config copy; the revision pin
# survives untouched. NOTE: notebook cells never import the tuned package - a
# kernel started before the editable install misses the .pth; only
# subprocesses see it. Plain yaml everywhere in this notebook.
import yaml
from pathlib import Path
from huggingface_hub import HfApi, create_repo

_raw = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
_name = _raw["hub"]["checkpoint_repo"].split("/", 1)[1]
CKPT_REPO = f"{HfApi().whoami()['name']}/{_name}"
_raw["hub"]["checkpoint_repo"] = CKPT_REPO
Path("configs/law_v1_run.yaml").write_text(yaml.safe_dump(_raw, sort_keys=False), encoding="utf-8")
CONFIG = "configs/law_v1_run.yaml"
create_repo(CKPT_REPO, private=True, exist_ok=True)  # exists before the first progress push
print(f"checkpoint repo for this run: {CKPT_REPO}")


In [ ]:
import subprocess

# CONFIG, not a hardcoded path: the dataset's think tags must match the model
# the config trains (<think> for Qwen3), and the re-home cell above swapped
# CONFIG for the session-local copy. Both builds carry hard timeouts: the
# smoke child once finished its work but hung at interpreter shutdown
# (abandoned streaming iterator; smoke.py now os._exit(0)s), and an
# unbounded cell wedges the whole session (v8 lesson, and likely v9's real
# stall). Healthy smoke build is ~11s (v9 log); 20 min is a generous ceiling.
assert subprocess.run(["python", "-m", "tuned.data.smoke", "--config", CONFIG], timeout=20 * 60).returncode == 0, "dataset build failed"
if MODE == "PROBE":
    # Long multi-turn examples so probe batches really reach the probed seq -
    # unpacked short examples would make the VRAM probe a false green. Target
    # defaults to the config's max_seq_length; PROBE_SEQ probes above it.
    _probe_cmd = ["python", "-m", "tuned.data.probe", "--config", CONFIG]
    if globals().get("PROBE_SEQ"):
        _probe_cmd += ["--target-tokens", str(PROBE_SEQ)]
    assert subprocess.run(_probe_cmd, timeout=5 * 60).returncode == 0, "probe dataset build failed"


In [ ]:
# Model acquisition, in ITS OWN cell (Kaggle batch flushes output per
# completed cell, so this phase and its duration are visible).
# Fast path: a snapshot staged by notebooks/stage_model.ipynb and attached as
# an Input mounts under /kaggle/input/*/qwen3-8b-staged/ - local-disk reads,
# immune to the hub-stall class that killed v6-v9. Its REVISION.txt must
# match the config pin, else we IGNORE it (staleness guard) and fall back to
# the hub download in a subprocess with a HARD TIMEOUT (v8 lesson: an
# unbounded download can burn the whole session; 25 min is ~5x healthy).
# HF_XET_HIGH_PERFORMANCE is the hub>=1.x xet-native turbo (the old
# HF_HUB_ENABLE_HF_TRANSFER is a silent no-op there); scoped to the child.
import os, subprocess, sys, time
import yaml
from pathlib import Path

_m = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))["model"]
_staged = None
for _marker in Path("/kaggle/input").glob("*/qwen3-8b-staged/REVISION.txt"):
    _rec = _marker.read_text().split()
    if _rec and _rec[-1] == _m["revision"]:
        _staged = _marker.parent
        break
    print(f"staged snapshot {_marker.parent} revision {_rec[-1] if _rec else '?'} "
          f"!= config pin {_m['revision']} - ignoring it")
if _staged is not None:
    os.environ["TUNED_MODEL_PATH"] = str(_staged)  # sft.py loads by path, no network
    print(f"using staged snapshot {_staged} - hub download skipped")
else:
    _code = (
        "from huggingface_hub import snapshot_download\n"
        f"p = snapshot_download({_m['repo']!r}, revision={_m['revision']!r})\n"
        "print('snapshot at', p)\n"
    )
    _t0 = time.time()
    r = subprocess.run([sys.executable, "-u", "-c", _code], timeout=25 * 60,
                       env={**os.environ, "HF_XET_HIGH_PERFORMANCE": "1"})
    assert r.returncode == 0, "model download failed"
    print(f"model snapshot ready in {time.time() - _t0:.0f}s")

In [ ]:
# Supervisor instead of `!`: on Kaggle batch, a cell's output is only flushed
# when the cell COMPLETES (cancelling discards it - never cancel this cell; let
# the watchdog kill and flush). Popen on plain pipes streams line-by-line, tees
# to a persisted log, pushes that log to the HF ckpt repo every 5 min for live
# remote visibility, heartbeats with GPU stats during silence, and hard-fails
# the notebook on a non-zero exit.
import os, shlex, signal, subprocess, sys, threading, time
from pathlib import Path

import yaml
from huggingface_hub import HfApi

PROBE_SEQ = globals().get("PROBE_SEQ", None)  # stale cell-1 tolerance
# RESUME extends max_steps past SMOKE's 60: the final checkpoint sits AT
# max_steps, so a bare --resume would load it and exit without a single step -
# a no-op false green. 4 extra steps force a real optimizer/scaler/rng reload;
# green = first logged step is 61, not 1. sft.py refuses a resume whose
# max_steps differs from the checkpoint state unless --allow-schedule-change
# is passed: warmup and the LR decay denominator are rebuilt from max_steps,
# so the LR jumps at the resume step - fine for a 4-step gate, never for the
# main run.
# MAIN / MAIN_RESUME are the production entries: --mode main selects the
# config's train.main block (ga=6, save_steps=10, data/law_v1.jsonl) and the
# clean-stop budget 37800 s = 10.5 h measured from process start - ~30 min
# inside this cell's 11 h watchdog, so _TimeBudget checkpoints and exits rc=0
# BEFORE any kill signal. They must NEVER carry --allow-schedule-change: on a
# production resume that jump is the +134% LR bug the schedule guard exists
# to prevent. sft.py refuses MAIN while train.main.max_steps is still the
# underived 0 sentinel.
ARGS = {
    # One gate, not two: the checkpoint-push path does not depend on row
    # length, so it rides along on the probe dataset instead of costing a
    # second model load. --save-steps 1 pushes at step 1; dropping --no-hub
    # is what makes it a save gate. This also proves a push succeeds AT the
    # new sequence length, which the split PROBE/SAVETEST never tested.
    "PROBE": ["--max-steps", "2", "--save-steps", "1",
              "--dataset", "data/probe_long.jsonl"]
             + (["--max-seq-length", str(PROBE_SEQ)] if PROBE_SEQ else []),
    "SMOKE": [],
    "RESUME": ["--resume", "--max-steps", "64", "--allow-schedule-change"],
    "MAIN": ["--time-budget-s", "37800"],
    "MAIN_RESUME": ["--resume", "--time-budget-s", "37800"],
}
if MODE not in ARGS:
    raise ValueError(f"unknown MODE {MODE!r}")

# Belt-and-suspenders: a stale kernel whose dataset cell predates PROBE would
# otherwise die 55s into the child with FileNotFoundError (the 2026-08-07
# lesson). Build here if missing - the child depends on it.
if MODE == "PROBE" and not Path("data/probe_long.jsonl").exists():
    print("probe dataset missing - building it now")
    _probe_cmd = ["python", "-m", "tuned.data.probe", "--config", CONFIG]
    if PROBE_SEQ:
        _probe_cmd += ["--target-tokens", str(PROBE_SEQ)]
    assert subprocess.run(_probe_cmd, timeout=5 * 60).returncode == 0, "probe dataset build failed"

# Same belt-and-suspenders for MAIN: spawning without the real dataset dies
# ~55 s into the child; fail in milliseconds instead. No auto-build here -
# the law_v1 builder has its own gates and must never run implicitly.
if MODE.startswith("MAIN"):
    _main_ds = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))["train"]["main"]["dataset"]
    assert Path(_main_ds).exists(), (
        f"main dataset {_main_ds} missing - build and gate it before launching MAIN"
    )

# torchrun spawns one rank per GPU; unsloth places rank N on cuda:N (needs the
# 0,1 visibility set in cell 1). Each rank holds the FULL model - the
# parallelism is over data, not layers. PYTHONUNBUFFERED below replaces
# `python -u`.
launcher = ["torchrun", "--nproc_per_node=2"]
mode_flag = "main" if MODE.startswith("MAIN") else "smoke"
cmd = [*launcher, "-m", "tuned.train.sft",
       "--config", CONFIG, "--mode", mode_flag, *ARGS[MODE]]
LOG_PATH = "/kaggle/working/train.log"   # persisted output dir - survives the session
# CKPT_REPO was set by the re-home cell above.
HEARTBEAT_S = 60
PUSH_EVERY_S = 300
# Model is pre-downloaded by the previous cell, so 45 min covers load + LoRA
# attach + 2 steps + 1 checkpoint push with slack; a stall now flushes its
# evidence in <=45 min instead of burning 2 h blind. PROBE: 2 steps at long
# seq plus the model load and the push fit the same budget. MAIN*:
# --time-budget-s stops cleanly ~30 min before this last-resort kill would
# fire.
TIMEOUT_S = 45 * 60 if MODE == "PROBE" else 11 * 3600

def announce(msg):
    line = msg.rstrip("\n") + "\n"
    print(line, end="", flush=True)          # notebook / iopub channel
    try:
        os.write(2, line.encode())           # raw fd channel (flushed with the cell)
    except OSError:
        pass

child_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
# start_new_session: WITHOUT it the child shares the Jupyter KERNEL's process
# group - any killpg would take the kernel down with it, and Kaggle batch
# discards a dead cell's buffered output (v6/v7 lesson). With it, killpg is
# scoped to the launcher's own subtree.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env=child_env, start_new_session=True)  # stderr merged: one ordered stream
announce(f"training child spawned pid={proc.pid} cmd={shlex.join(cmd)} tee={LOG_PATH}")

last_line = [time.monotonic()]              # last time the child said anything

def pump():
    with open(LOG_PATH, "ab", buffering=0) as log:   # unbuffered: survives SIGKILL
        for raw in iter(proc.stdout.readline, b""):
            log.write(raw)
            last_line[0] = time.monotonic()
            sys.stdout.write(raw.decode("utf-8", "replace"))
            sys.stdout.flush()
    proc.stdout.close()

pump_t = threading.Thread(target=pump, daemon=True)
pump_t.start()

def _rank_pids(parent):
    """Direct children of the launcher = the rank workers. torchrun spawns
    each rank with start_new_session=True (pytorch subprocess_handler), so
    they are NOT in the launcher's process group - killing that group alone
    would orphan two processes holding ~13 GiB of VRAM each. /proc scan, no
    psutil dependency."""
    pids = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open(f"/proc/{entry}/stat", "rb") as fh:
                fields = fh.read().rsplit(b") ", 1)[1].split()  # comm may contain spaces
            if int(fields[1]) == parent:                         # fields[1] = ppid
                pids.append(int(entry))
        except (OSError, IndexError, ValueError):
            continue
    return pids

def _killpg(pgid, sig):
    try:
        os.killpg(pgid, sig)
    except (ProcessLookupError, PermissionError):
        pass

def terminate_child(ranks, grace_s=60):
    """SIGTERM the launcher and let torchrun reap its own workers: its elastic
    agent catches SIGTERM and runs killpg(TERM) -> 30 s -> killpg(KILL) per
    rank; grace_s covers that escalation. Then SIGKILL our group and each
    surviving rank (each rank is its own session leader: pgid == pid). A
    longer grace does NOT save an in-flight checkpoint push - SIGTERM kills a
    rank outright (no handler installed) - and Hub commits are atomic, so the
    repo stays consistent either way."""
    _killpg(os.getpgid(proc.pid), signal.SIGTERM)
    try:
        proc.wait(timeout=grace_s)
    except subprocess.TimeoutExpired:
        announce(f"[watchdog] launcher alive {grace_s}s after SIGTERM - SIGKILL")
        _killpg(os.getpgid(proc.pid), signal.SIGKILL)
    for pid in ranks:
        _killpg(pid, signal.SIGKILL)

_push_inflight = threading.Event()

def push_progress(elapsed):
    """Tee the log to the ckpt repo - the only channel visible mid-run.
    Uploads in a daemon thread: a slow or hung upload must never block THIS
    loop, which owns the heartbeats and the watchdog timeout check. Skips if
    the previous push is still in flight. Returns the thread or None."""
    if _push_inflight.is_set():
        announce(f"[progress-push skipped +{elapsed:.0f}s - previous push still in flight]")
        return None
    _push_inflight.set()

    def _upload():
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(
                path_or_fileobj=LOG_PATH, path_in_repo="progress/train.log",
                repo_id=CKPT_REPO, commit_message=f"progress +{elapsed:.0f}s",
            )
        except Exception as exc:
            announce(f"[progress-push failed] {exc!r}")
        finally:
            _push_inflight.clear()

    t = threading.Thread(target=_upload, daemon=True)
    t.start()
    return t

start = time.monotonic()
last_beat = start
last_push = start
ranks = []
try:
    while proc.poll() is None:
        time.sleep(5)
        # refresh while the launcher lives: once it exits the ppid link is
        # gone and any orphan becomes unfindable
        ranks = _rank_pids(proc.pid) or ranks
        now = time.monotonic()
        if now - start > TIMEOUT_S:
            announce(f"[watchdog] timeout after {now - start:.0f}s - terminating pid={proc.pid}")
            terminate_child(ranks)
            break
        if now - last_line[0] >= HEARTBEAT_S and now - last_beat >= HEARTBEAT_S:
            try:
                gpu = subprocess.run(
                    ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,"
                     "clocks.sm,clocks.max.sm,temperature.gpu,power.draw",
                     "--format=csv,noheader"],
                    capture_output=True, text=True, timeout=20,
                ).stdout.strip().replace("\n", " | ")
            except Exception as exc:
                gpu = f"nvidia-smi failed: {exc!r}"
            announce(f"[heartbeat +{now - start:.0f}s] pid={proc.pid} "
                     f"silent {now - last_line[0]:.0f}s; gpu: {gpu}")
            last_beat = now
        if now - last_push >= PUSH_EVERY_S:
            push_progress(now - start)
            last_push = now
finally:
    # A cell exception must not orphan the ranks: start_new_session means
    # they no longer die with the kernel. Idempotent - already-reaped pids
    # just miss (errors swallowed in _killpg).
    if proc.poll() is None:
        terminate_child(ranks)
    for pid in ranks:
        _killpg(pid, signal.SIGKILL)

pump_t.join(timeout=30)
rc = proc.wait()
announce(f"training child exited rc={rc} after {time.monotonic() - start:.0f}s")
final_push = push_progress(time.monotonic() - start)  # final log snapshot, wins even on failure
if final_push is not None:
    final_push.join(timeout=120)  # bounded: a hung upload must not wedge the cell either
if rc != 0:
    raise RuntimeError(f"training failed with exit code {rc} - see {LOG_PATH}")

## Green means
The gates below ran green on 2026-08-08 at seq 8192; the numbers are the reference to compare a re-run against, and the cap moved to 12288 on 2026-08-26.
- **PROBE** (2 steps at the requalified seq, pushes a checkpoint): green is
  `peak_vram_reserved_gb_dev0/1` below **13.5 GiB** with >= 1 GiB margin on the
  worst rank, finite loss, `label_coverage=` nonzero, and `last-checkpoint/`
  visible in the checkpoint repo printed by the re-home cell. `eos_in_labels=`
  near-zero is the EXPECTED artifact of truncating probe rows, not a failure.
  `grad_norm=nan` on steps 1-2 is GradScaler calibration, benign. Reference
  peaks at seq 8192 were 12.98/13.18 GiB; the seq 12288 reference is recorded
  here after the first green run. If OOM, walk the ladder in
  `configs/law_v1_8b_ddp.yaml`: standard-quant repo (-1.31 GiB) -> seq 8192 ->
  seq 6144. n_chunks is already at 32. No CPU-offload checkpointing rescue
  exists under DDP.
- **SMOKE**: 60 steps complete, loss trending down, **no NaN** (fp16 canary),
  `peak_vram_gb` < 14, ~1.2 h at ~74.7 s/step. Record `approx_tokens_per_sec` and
  total session hours for the main-run plan. Reference run: train_loss 0.5722 (down),
  grad_norm finite after 2-step calibration, peaks 12.98/13.18 GiB
  (~219 tok/s/rank upper bound).
- **RESUME**: run in a *fresh* session; `--max-steps 64` extends past SMOKE's final
  step-60 checkpoint (which sits AT max_steps - a bare `--resume` would no-op into a
  false green). Green = the first logged step is **61**, not 1, and 4 steps complete
  with finite grad_norm.
- **MAIN / MAIN_RESUME** (production, 2026-08-09 wiring): `--mode main` runs the
  config's `train.main` block - ga=6 (~3x longer optimizer step, ~224 s expected),
  save_steps=10 (~37 min cadence), `data/law_v1.jsonl`. Prereqs sft.py enforces:
  `train.main.max_steps` derived from the `post_filter_rows=` line of a 2-step
  `--no-hub` probe and committed (0 sentinel refuses to train), dataset built.
  `--time-budget-s 37800` checkpoints and exits rc=0 at 10.5 h from process start -
  a MAIN session ending by watchdog kill is a bug, not normal operation. NEVER add
  `--allow-schedule-change` to a MAIN entry; never edit max_steps between sessions.
- **2026-08-09 hardening additions** (this validation pass exists to exercise them):
  `label_coverage=<n>/<total>` must print nonzero before step 1 (a SystemExit there is
  the step-0 gate catching an all-masked batch, not a crash); `eos_in_labels=` must
  print nonzero too (warn-only on truncating smoke data, fatal on main - a model that
  never sees `<|im_end|>` in labels never learns to stop); pad positions carrying
  labels abort (the pad IS `<|endoftext|>`); dataset prep now runs rank-0-first
  (`local_main_process_first`) - ONE tokenization bar, not two interleaved ones, is
  the new normal; record `peak_vram_reserved_gb` beside the allocated peaks -
  reserved is what actually OOMs and the ~13.5 abort line is enforced against it;
  RESUME still requires `--allow-schedule-change` (already in ARGS) because sft.py
  refuses a resume whose max_steps differs from the checkpoint's; the resume-session
  final `train_loss` summary reads ~10x low (running-loss reset on resume) - read the
  per-step losses instead.
- DDP sanity: both unsloth banners must say **Num GPUs = 2**; "Num GPUs = 1"
  means the single-GPU mask leaked (sft.py fails fast on it since the 2026-08-06 crash).
- Gate ladder: PROBE (merged save gate) -> SMOKE -> RESUME, each green before the next.
  Requalification for the 2026-08-09 changes (ga=6 main block, group_by_length,
  drop_last, new gates, rank-0-first prep): memory shape is provably unchanged
  (same bs, same bucket; ga is sequential micro-batches), so a SAVETEST-first
  short ladder is acceptable - but the supervisor/kill-path changes make one full
  SMOKE before the main run the safer call.
- Note: SAVETEST touches only ~64 examples - an OOM later in the full SMOKE run is still
  possible; watch peak_vram_reserved_gb (this lane runs ~1.4 GiB from the cap on the
  old decimal accounting, ~2.3 GiB in real GiB).
- **Never cancel the training cell.** Kaggle batch discards a cancelled cell's buffered
  output (v6/v7 lesson) - the watchdog kills and flushes on its own (SIGTERM first so
  torchrun reaps its rank workers, then SIGKILL + a /proc sweep for survivors; orphaned
  ranks would silently hold ~13 GiB of VRAM each). Mid-run visibility:
  `progress/train.log` in the HF checkpoint repo, refreshed every 5 min.